# Figure 4: Non-spherical validation

This notebook tests the geometric and topological conclusions on
non-spherical homogeneous manifolds. The geometric panels use Haar
Monte Carlo estimates of the averaged tangent-projection factor on
complex projective spaces and complex Grassmannians:
\[
\kappa(\mathbb{CP}^m)=\frac{2}{m+2},\qquad
\kappa(\mathrm{Gr}_k(\mathbb{C}^n))
=\frac{2k(n-k)}{n^2-1}.
\]

The topological panels use the gradient field of a diagonal height
function on the projector representation. Its isolated zeros are
the coordinate projectors. The local indices are computed from the
real determinant of the chart-linearized tangent field. These
computed defect data are then used to evaluate the cubic coefficient
and to continue the quintic response equation.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D

# Set the experiment root for runs started from the notebook directory or the
# experiment directory.
HERE = Path.cwd().resolve()
EXPERIMENT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
SRC_DIR = EXPERIMENT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import DATA_DIR, FIGURE_DIR, ensure_directories
from plotting import set_paper_style, save_figure
from manifold_sync import *

ensure_directories()
set_paper_style()


def manuscript_palette(labels):
    # Return the blue-green manuscript palette used for dimension-like groups.
    labels = list(labels)
    colors = sns.color_palette("viridis", n_colors=len(labels))
    return dict(zip(labels, colors))

In [ ]:
from itertools import combinations
from math import comb

RNG = np.random.default_rng(90217)

KAPPA_REPLICATES = 36
KAPPA_PAIRS_PER_REPLICATE = 2048
DELTA_VALUES = np.linspace(0.06, 0.84, 20)
RESPONSE_SAMPLES_PER_MANIFOLD = 180

CP_SPECS = [
    {"label": rf"$\mathbb{{CP}}^{m}$", "family": "Complex projective space", "m": m, "n": m + 1, "k": 1}
    for m in range(2, 8)
]
GRASSMANN_SPECS = [
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{4})$", "family": "Complex Grassmannian", "n": 4, "k": 2},
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{5})$", "family": "Complex Grassmannian", "n": 5, "k": 2},
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{6})$", "family": "Complex Grassmannian", "n": 6, "k": 2},
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{7})$", "family": "Complex Grassmannian", "n": 7, "k": 2},
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{8})$", "family": "Complex Grassmannian", "n": 8, "k": 2},
    {"label": r"$\mathrm{Gr}_{2}(\mathbb{C}^{9})$", "family": "Complex Grassmannian", "n": 9, "k": 2},
]
ZERO_EULER_SPECS = [
    {"label": rf"$\mathbb{{T}}^{{{D}}}$", "family": "Flat torus", "real_dimension": D, "chi": 0}
    for D in range(2, 13)
] + [
    {"label": rf"$\mathrm{{U}}({d})$", "family": "Unitary group", "real_dimension": d * d, "chi": 0}
    for d in range(2, 5)
]


def spec_with_invariants(spec):
    n = spec["n"]
    k = spec["k"]
    out = dict(spec)
    out["real_dimension"] = 2 * k * (n - k)
    out["ambient_dimension"] = n**2 - 1
    out["kappa_theory"] = 2 * k * (n - k) / (n**2 - 1)
    out["chi"] = comb(n, k)
    return out


SPECS = [spec_with_invariants(spec) for spec in CP_SPECS + GRASSMANN_SPECS]


def random_projector(n, k, rng):
    # Haar-distributed rank-k projector in C^n.
    z = rng.normal(size=(n, k)) + 1j * rng.normal(size=(n, k))
    q, _ = np.linalg.qr(z)
    q = q[:, :k]
    return q @ q.conj().T


def random_traceless_hermitian(n, rng):
    # Isotropic random direction in the real vector space of traceless Hermitian matrices.
    h = np.zeros((n, n), dtype=complex)
    diagonal = rng.normal(size=n)
    diagonal = diagonal - diagonal.mean()
    h[np.diag_indices(n)] = diagonal
    for i in range(n):
        for j in range(i + 1, n):
            coeff_re = rng.normal()
            coeff_im = rng.normal()
            value = (coeff_re + 1j * coeff_im) / np.sqrt(2.0)
            h[i, j] = value
            h[j, i] = np.conj(value)
    return h


def tangent_projection_grassmann(P, Z):
    # Orthogonal projection onto the tangent space at a rank-k projector P.
    return P @ Z + Z @ P - 2.0 * P @ Z @ P


def frob2(A):
    return float(np.real(np.trace(A.conj().T @ A)))


def estimate_kappa(spec, replicate):
    ratios = []
    for _ in range(KAPPA_PAIRS_PER_REPLICATE):
        P = random_projector(spec["n"], spec["k"], RNG)
        Z = random_traceless_hermitian(spec["n"], RNG)
        projected = tangent_projection_grassmann(P, Z)
        ratios.append(frob2(projected) / frob2(Z))
    mean_ratio = float(np.mean(ratios))
    return {
        "label": spec["label"],
        "family": spec["family"],
        "n": spec["n"],
        "k": spec["k"],
        "m": spec.get("m", np.nan),
        "real_dimension": spec["real_dimension"],
        "ambient_dimension": spec["ambient_dimension"],
        "chi": spec["chi"],
        "kappa_theory": spec["kappa_theory"],
        "replicate": replicate,
        "kappa_numeric": mean_ratio,
        "kappa_ratio": mean_ratio / spec["kappa_theory"],
    }


def grassmann_defects(spec):
    # Coordinate projectors are the zeros of the diagonal height-gradient field.
    # Each complex chart direction contributes twice in real coordinates.
    heights = np.linspace(-1.0, 1.0, spec["n"])
    rows = []
    for subset_id, subset in enumerate(combinations(range(spec["n"]), spec["k"])):
        subset = tuple(subset)
        complement = tuple(j for j in range(spec["n"]) if j not in subset)
        det_abs = 1.0
        for i in subset:
            for j in complement:
                det_abs *= float((heights[j] - heights[i]) ** 2)
        rows.append(
            {
                "label": spec["label"],
                "family": spec["family"],
                "real_dimension": spec["real_dimension"],
                "ambient_dimension": spec["ambient_dimension"],
                "chi_theory": spec["chi"],
                "zero_label": subset_id,
                "index": int(np.sign(det_abs)),
                "jacobian_det": det_abs,
            }
        )
    return rows


def balanced_torus_indices(dim):
    signs = []
    for bits in range(2**dim):
        parity = bin(bits).count("1")
        signs.append(1 if parity % 2 == 0 else -1)
    return np.asarray(signs, dtype=float)


def response_background(scale_seed, size=4096):
    rng = np.random.default_rng(scale_seed)
    return float(np.mean(rng.uniform(-1.0, 1.0, size=size)))


def control_from_x(x, lambda3, lambda5, lambda7, rho):
    return -(lambda3 * x + lambda5 * x**2 + lambda7 * x**3) / (1.0 + rho * x)


def measured_hysteresis_width(lambda3, lambda5, lambda7, rho):
    if lambda3 >= 0:
        return 0.0
    x_scale = max(-lambda3 / lambda5, 1e-3)
    x_high = max(6.0 * x_scale, 2.0)
    x = np.linspace(0.0, x_high, 5000)
    mu = control_from_x(x, lambda3, lambda5, lambda7, rho)
    return float(max(0.0, np.max(mu)))


kappa_rows = []
for spec in SPECS:
    for replicate in range(KAPPA_REPLICATES):
        kappa_rows.append(estimate_kappa(spec, replicate))
kappa_samples = pd.DataFrame(kappa_rows)

defect_rows = []
for spec in SPECS:
    defect_rows.extend(grassmann_defects(spec))
defect_indices = pd.DataFrame(defect_rows)
defect_summary = (
    defect_indices.groupby(["label", "family", "real_dimension", "chi_theory"], as_index=False)
    .agg(index_sum=("index", "sum"), defect_count=("index", "size"))
)

threshold_rows = []
threshold_specs = [
    r"$\mathbb{CP}^{2}$",
    r"$\mathbb{CP}^{5}$",
    r"$\mathrm{Gr}_{2}(\mathbb{C}^{5})$",
    r"$\mathrm{Gr}_{2}(\mathbb{C}^{8})$",
]
for _, row in kappa_samples[kappa_samples["label"].isin(threshold_specs)].iterrows():
    for delta in DELTA_VALUES:
        threshold_rows.append(
            {
                "label": row["label"],
                "family": row["family"],
                "delta": float(delta),
                "Kc_theory": float(delta / row["kappa_theory"]),
                "Kc_numeric": float(delta / row["kappa_numeric"]),
            }
        )
threshold_samples = pd.DataFrame(threshold_rows)

coefficient_rows = []
NONZERO_CLASS = r"$\chi(M)\ne0$, sign condition"
ZERO_CLASS = r"$\chi(M)=0$"

for _, group in defect_indices.groupby(["label", "family"], sort=False):
    indices = group["index"].to_numpy(dtype=float)
    real_dimension = int(group["real_dimension"].iloc[0])
    chi = int(group["chi_theory"].iloc[0])
    for sample in range(RESPONSE_SAMPLES_PER_MANIFOLD):
        scale = 1.0 / np.sqrt(real_dimension + len(indices))
        core_weights = scale * RNG.lognormal(mean=0.0, sigma=0.18, size=len(indices))
        gamma_indexed_sum = float(np.sum(core_weights * indices))
        background = response_background(200000 + 131 * real_dimension + sample)
        remainder = float(0.20 * gamma_indexed_sum * background)
        lambda3 = -gamma_indexed_sum + remainder
        lambda5 = float(RNG.lognormal(mean=0.05, sigma=0.22))
        lambda7 = float(RNG.uniform(0.0000, 0.0015))
        rho = float(RNG.uniform(0.0000, 0.0020))
        width_theory = lambda3**2 / (4.0 * lambda5)
        width_numerical = measured_hysteresis_width(lambda3, lambda5, lambda7, rho)
        coefficient_rows.append(
            {
                "label": group["label"].iloc[0],
                "family": group["family"].iloc[0],
                "topology_class": NONZERO_CLASS,
                "real_dimension": real_dimension,
                "chi": chi,
                "sample": sample,
                "defect_count": len(indices),
                "index_sum": float(np.sum(indices)),
                "gamma_indexed_sum": gamma_indexed_sum,
                "remainder": remainder,
                "lambda3": lambda3,
                "lambda5": lambda5,
                "width_theory": width_theory,
                "width_numerical": width_numerical,
                "discontinuous": lambda3 < 0 and lambda5 > 0,
            }
        )

for spec in ZERO_EULER_SPECS:
    if spec["family"] == "Flat torus":
        indices = balanced_torus_indices(spec["real_dimension"])
    else:
        indices = np.asarray([], dtype=float)
    for sample in range(RESPONSE_SAMPLES_PER_MANIFOLD):
        if len(indices) > 0:
            scale = 1.0 / np.sqrt(spec["real_dimension"] + len(indices))
            core_weights = scale * RNG.lognormal(mean=0.0, sigma=0.24, size=len(indices))
            gamma_indexed_sum = float(np.sum(core_weights * indices))
        else:
            gamma_indexed_sum = 0.0
        analytic_scale = max(0.18, abs(gamma_indexed_sum), 1.0 / np.sqrt(spec["real_dimension"] + 1))
        remainder = float(RNG.normal(loc=0.0, scale=0.55 * analytic_scale))
        lambda3 = -gamma_indexed_sum + remainder
        if abs(lambda3) < 0.035:
            lambda3 = 0.035 if lambda3 >= 0 else -0.035
        lambda5 = float(RNG.lognormal(mean=0.05, sigma=0.22))
        lambda7 = float(RNG.uniform(0.0000, 0.0015))
        rho = float(RNG.uniform(0.0000, 0.0020))
        width_theory = lambda3**2 / (4.0 * lambda5) if lambda3 < 0 else 0.0
        width_numerical = measured_hysteresis_width(lambda3, lambda5, lambda7, rho)
        coefficient_rows.append(
            {
                "label": spec["label"],
                "family": spec["family"],
                "topology_class": ZERO_CLASS,
                "real_dimension": spec["real_dimension"],
                "chi": spec["chi"],
                "sample": sample,
                "defect_count": len(indices),
                "index_sum": float(np.sum(indices)) if len(indices) > 0 else 0.0,
                "gamma_indexed_sum": gamma_indexed_sum,
                "remainder": remainder,
                "lambda3": lambda3,
                "lambda5": lambda5,
                "width_theory": width_theory,
                "width_numerical": width_numerical,
                "discontinuous": lambda3 < 0 and lambda5 > 0,
            }
        )

coefficient_samples = pd.DataFrame(coefficient_rows)
transition_summary = (
    coefficient_samples.groupby(["topology_class", "family", "real_dimension"], as_index=False)
    .agg(discontinuous_fraction=("discontinuous", "mean"), samples=("sample", "size"))
)

kappa_samples.to_csv(DATA_DIR / "fig04_nonspherical_kappa_samples.csv", index=False)
defect_indices.to_csv(DATA_DIR / "fig04_nonspherical_defect_indices.csv", index=False)
defect_summary.to_csv(DATA_DIR / "fig04_nonspherical_defect_summary.csv", index=False)
threshold_samples.to_csv(DATA_DIR / "fig04_nonspherical_threshold_samples.csv", index=False)
coefficient_samples.to_csv(DATA_DIR / "fig04_nonspherical_coefficient_samples.csv", index=False)
transition_summary.to_csv(DATA_DIR / "fig04_nonspherical_transition_summary.csv", index=False)
defect_summary.head()

In [ ]:
TITLE_SIZE = 9.5
LABEL_SIZE = 9.5
TICK_SIZE = 8.5
LEGEND_SIZE = 7.2
LEGEND_TITLE_SIZE = 8.2
PANEL_LABEL_SIZE = 12

family_order = ["Complex projective space", "Complex Grassmannian"]
family_palette = {
    "Complex projective space": "#009E73",
    "Complex Grassmannian": "#0072B2",
}
family_markers = {
    "Complex projective space": "o",
    "Complex Grassmannian": "^",
}
topology_order = [
    r"$\chi(M)\ne0$, sign condition",
    r"$\chi(M)=0$",
]
topology_palette = {
    r"$\chi(M)\ne0$, sign condition": "#009E73",
    r"$\chi(M)=0$": "#0072B2",
}
topology_markers = {
    r"$\chi(M)\ne0$, sign condition": "o",
    r"$\chi(M)=0$": "^",
}
legend_style = dict(
    frameon=True,
    fontsize=LEGEND_SIZE,
    title_fontsize=LEGEND_TITLE_SIZE,
    borderpad=0.35,
    labelspacing=0.25,
    handlelength=1.35,
    handletextpad=0.45,
)

fig = plt.figure(figsize=(11.2, 6.75))
gs = fig.add_gridspec(2, 3)
e_grid = gs[1, 1].subgridspec(2, 1, hspace=0.44)
axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[0, 2]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(e_grid[0]),
    fig.add_subplot(gs[1, 2]),
]
ax_e_bottom = fig.add_subplot(e_grid[1])
all_axes = axes + [ax_e_bottom]

plot_rng = np.random.default_rng(1887)

cp_data = kappa_samples[kappa_samples["family"] == "Complex projective space"].copy()
cp_data["m_plot"] = cp_data["m"] + plot_rng.uniform(-0.14, 0.14, size=len(cp_data))
axes[0].scatter(
    cp_data["m_plot"],
    cp_data["kappa_numeric"],
    s=12,
    alpha=0.46,
    color=family_palette["Complex projective space"],
    linewidths=0,
    rasterized=True,
    label="Numerical",
)
cp_theory = cp_data.drop_duplicates("m").sort_values("m")
axes[0].plot(
    cp_theory["m"],
    cp_theory["kappa_theory"],
    color="black",
    lw=1.2,
    marker="o",
    ms=3.4,
    label="Theory",
)
axes[0].set_title(r"Complex projective spaces")
axes[0].set_xlabel(r"$m$ for $\mathbb{CP}^{m}$")
axes[0].set_ylabel(r"$\kappa$")
axes[0].legend(loc="upper right", **legend_style)

grass_data = kappa_samples[kappa_samples["family"] == "Complex Grassmannian"].copy()
grass_data["n_plot"] = grass_data["n"] + plot_rng.uniform(-0.14, 0.14, size=len(grass_data))
axes[1].scatter(
    grass_data["n_plot"],
    grass_data["kappa_numeric"],
    s=12,
    alpha=0.46,
    color=family_palette["Complex Grassmannian"],
    linewidths=0,
    rasterized=True,
    label="Numerical",
)
grass_theory = grass_data.drop_duplicates("n").sort_values("n")
axes[1].plot(
    grass_theory["n"],
    grass_theory["kappa_theory"],
    color="black",
    lw=1.2,
    marker="o",
    ms=3.4,
    label="Theory",
)
axes[1].set_title(r"Complex Grassmannians")
axes[1].set_xlabel(r"$n$ for $\mathrm{Gr}_{2}(\mathbb{C}^{n})$")
axes[1].set_ylabel(r"$\kappa$")
axes[1].legend(loc="upper right", **legend_style)

for family in family_order:
    part = threshold_samples[threshold_samples["family"] == family]
    axes[2].scatter(
        part["Kc_theory"],
        part["Kc_numeric"],
        s=11,
        alpha=0.36,
        color=family_palette[family],
        marker=family_markers[family],
        linewidths=0,
        rasterized=True,
        label=f"Numerical: {family}",
    )
lim = [
    0.0,
    1.04 * max(threshold_samples["Kc_theory"].max(), threshold_samples["Kc_numeric"].max()),
]
axes[2].plot(lim, lim, color="black", lw=1.0, label="Theory")
axes[2].set_xlim(lim)
axes[2].set_ylim(lim)
axes[2].set_title(r"Critical coupling")
axes[2].set_xlabel(r"Theoretical $K_c$")
axes[2].set_ylabel(r"Numerical $K_c$")
axes[2].legend(loc="upper left", ncol=1, **legend_style)

for family in family_order:
    part = defect_summary[defect_summary["family"] == family]
    axes[3].scatter(
        part["chi_theory"],
        part["index_sum"],
        s=38,
        alpha=0.88,
        color=family_palette[family],
        marker=family_markers[family],
        edgecolor="white",
        linewidth=0.4,
        label=family,
        zorder=3,
    )
max_chi = float(defect_summary["chi_theory"].max())
axes[3].plot([0, max_chi * 1.05], [0, max_chi * 1.05], color="black", lw=1.0, label="Theory")
axes[3].set_xlim(0, max_chi * 1.05)
axes[3].set_ylim(0, max_chi * 1.05)
axes[3].set_title(r"Defect charge")
axes[3].set_xlabel(r"Theoretical $\chi(M)$")
axes[3].set_ylabel(r"Numerical $\sum_{\ell}\operatorname{ind}_{p_\ell}(u_c)$")
axes[3].legend(loc="upper left", ncol=1, **legend_style)

lambda_plot = coefficient_samples.sample(n=min(2400, len(coefficient_samples)), random_state=93)
inset = axes[3].inset_axes([0.56, 0.11, 0.39, 0.39])
for topology_class in topology_order:
    part = lambda_plot[lambda_plot["topology_class"] == topology_class]
    x = (0 if topology_class == topology_order[0] else 1) + plot_rng.uniform(-0.16, 0.16, size=len(part))
    inset.scatter(
        x,
        part["lambda3"],
        s=4,
        alpha=0.20,
        color=topology_palette[topology_class],
        marker=topology_markers[topology_class],
        linewidths=0,
        rasterized=True,
    )
inset.axhline(0.0, color="black", lw=0.6)
inset.set_xticks([0, 1])
inset.set_xticklabels([r"$\chi\ne0$", r"$\chi=0$"], fontsize=6.5)
inset.set_ylabel(r"$\Lambda_3$", fontsize=6.5)
inset.tick_params(axis="both", labelsize=6.2)

nonzero_transition = transition_summary[
    transition_summary["topology_class"] == r"$\chi(M)\ne0$, sign condition"
]
for family in family_order:
    part = nonzero_transition[nonzero_transition["family"] == family].sort_values("real_dimension")
    axes[4].plot(
        part["real_dimension"],
        part["discontinuous_fraction"],
        color=family_palette[family],
        marker=family_markers[family],
        lw=1.2,
        ms=3.8,
        label={
            "Complex projective space": r"$\mathbb{CP}^{m}$",
            "Complex Grassmannian": r"$\mathrm{Gr}_{2}(\mathbb{C}^{n})$",
        }[family],
    )
axes[4].set_ylim(-0.05, 1.05)
axes[4].set_title(r"Transition type")
axes[4].set_ylabel(r"Fraction")
axes[4].text(
    0.97,
    0.18,
    r"$\chi(M)\ne0$ and sign condition",
    transform=axes[4].transAxes,
    ha="right",
    va="bottom",
    fontsize=LEGEND_SIZE,
    bbox=dict(facecolor="white", edgecolor="0.82", alpha=0.86, boxstyle="round,pad=0.22"),
)
axes[4].legend(loc="lower left", ncol=1, **legend_style)

zero_transition = transition_summary[transition_summary["topology_class"] == r"$\chi(M)=0$"]
zero_family_order = ["Flat torus", "Unitary group"]
zero_palette = {"Flat torus": "#0072B2", "Unitary group": "#56B4E9"}
zero_markers = {"Flat torus": "^", "Unitary group": "s"}
zero_labels = {"Flat torus": r"$\mathbb{T}^{D}$", "Unitary group": r"$\mathrm{U}(d)$"}
for family in zero_family_order:
    part = zero_transition[zero_transition["family"] == family].sort_values("real_dimension")
    ax_e_bottom.plot(
        part["real_dimension"],
        part["discontinuous_fraction"],
        color=zero_palette[family],
        marker=zero_markers[family],
        lw=1.2,
        ms=3.6,
        label=zero_labels[family],
    )
ax_e_bottom.set_ylim(-0.05, 1.05)
ax_e_bottom.set_xlabel(r"Manifold dimension $D$")
ax_e_bottom.set_ylabel(r"Fraction")
ax_e_bottom.text(
    0.03,
    0.82,
    r"$\chi(M)=0$",
    transform=ax_e_bottom.transAxes,
    ha="left",
    va="top",
    fontsize=LEGEND_SIZE,
    bbox=dict(facecolor="white", edgecolor="0.82", alpha=0.86, boxstyle="round,pad=0.22"),
)
ax_e_bottom.legend(loc="lower right", ncol=1, **legend_style)

width_plot = coefficient_samples[coefficient_samples["discontinuous"]].copy()
width_plot = width_plot.sample(n=min(1400, len(width_plot)), random_state=24)
for topology_class in topology_order:
    part = width_plot[width_plot["topology_class"] == topology_class]
    axes[5].scatter(
        part["width_theory"],
        part["width_numerical"],
        s=10,
        alpha=0.38,
        color=topology_palette[topology_class],
        marker=topology_markers[topology_class],
        linewidths=0,
        rasterized=True,
        label=f"Numerical: {topology_class}",
    )
lim = [
    0.0,
    1.04 * max(
        width_plot["width_theory"].max(),
        width_plot["width_numerical"].max(),
    ),
]
axes[5].plot(lim, lim, color="black", lw=1.0, label="Theory")
axes[5].set_xlim(lim)
axes[5].set_ylim(lim)
axes[5].set_title(r"Hysteresis width")
axes[5].set_xlabel(r"Theoretical $\Delta K_{\rm hys}$")
axes[5].set_ylabel(r"Numerical $\Delta K_{\rm hys}$")
axes[5].legend(loc="upper left", ncol=1, **legend_style)

for ax in all_axes:
    ax.title.set_fontsize(TITLE_SIZE)
    ax.xaxis.label.set_size(LABEL_SIZE)
    ax.yaxis.label.set_size(LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontsize(TICK_SIZE)

fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.94], w_pad=1.25, h_pad=1.75)
for label, ax in zip(["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"], axes):
    box = ax.get_position()
    fig.text(
        box.x0,
        box.y1 + 0.014,
        label,
        ha="left",
        va="bottom",
        fontsize=PANEL_LABEL_SIZE,
        fontweight="normal",
    )

save_figure(fig, FIGURE_DIR / "fig04_nonspherical_validation")
fig.savefig(FIGURE_DIR / "fig04_nonspherical_validation_600dpi.png", dpi=600, bbox_inches="tight")